# ScanNet++ NVS results (DSLR, ours_30000)

Test split is 3DGS `--eval` (every 8th registered image, sorted by name; test poses come
from each arm's own SfM). Render `000NN.png` = position NN of that arm's sorted test list —
arms can register different image sets, so views are re-keyed to **original image names**
via `cameras.json` and all comparisons use the per-scene intersection of test sets.

In [85]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

root = Path("/Users/adam/Documents/MILA/projects/depth-aware-ba/3dgs/scannetpp/dslr")
ITER = "ours_30000"
METRICS = ["PSNR", "SSIM", "LPIPS"]
BASELINE = "baseline"
ARMS = ["baseline", "unimodal", "gmm", "unimodal_all", "gmm_all", "unimodal_local", "gmm_local"]
T = 0.5        # dB win/loss threshold
LLFFHOLD = 8   # 3DGS --eval test split

def load_results(scene, arm):
    p = root / f"{scene}_{arm}" / "results.json"
    return json.load(open(p)).get(ITER) if p.exists() else None

def test_names(scene, arm):
    """Original image names of the arm's test views, in render order."""
    cams = json.load(open(root / f"{scene}_{arm}" / "cameras.json"))
    names = sorted(c["img_name"] for c in cams)
    return [n for i, n in enumerate(names) if i % LLFFHOLD == 0]

def per_view(scene, arm):
    p = root / f"{scene}_{arm}" / "per_view.json"
    if not p.exists():
        return None
    d = json.load(open(p))[ITER]
    df = pd.DataFrame({m: d[m] for m in METRICS})
    tn = test_names(scene, arm)
    assert len(df) == len(tn), f"{scene}/{arm}: {len(df)} views vs {len(tn)} test names"
    df.index = tn
    return df.rename_axis("img")

scene_ids = sorted({d.name.split("_", 1)[0] for d in root.iterdir() if d.is_dir()})
scenes = [s for s in scene_ids if all(load_results(s, a) for a in ARMS)]
print(f"scenes: {len(scenes)} / {len(scene_ids)}", scenes)
pd.set_option("display.width", 200)

scenes: 8 / 9 ['0d2ee665be', '13c3e046d7', '1ada7a0617', '21d970d8de', '25f3b7a318', '27dd4da69e', '286b55a2bf', '31a2c91c43']


## Integrity — registered images and common test views per arm

In [86]:
n_reg = pd.DataFrame(
    {a: {s: len(json.load(open(root / f"{s}_{a}" / "cameras.json"))) for s in scenes}
     for a in ARMS}
)
common_imgs = {}
for s in scenes:
    sets = [set(test_names(s, a)) for a in ARMS]
    common_imgs[s] = sorted(set.intersection(*sets))
n_reg["common_test/baseline_test"] = [
    f"{len(common_imgs[s])}/{len(test_names(s, BASELINE))}" for s in scenes
]
n_reg

,baseline,unimodal,gmm,unimodal_all,gmm_all,unimodal_local,gmm_local,common_test/baseline_test
0d2ee665be,182,182,182,182,182,182,182,23/23
13c3e046d7,418,418,418,418,418,418,418,53/53
1ada7a0617,343,343,343,343,343,340,343,30/43
21d970d8de,285,285,285,285,285,285,285,36/36
25f3b7a318,325,325,325,325,325,325,325,41/41
27dd4da69e,93,93,93,93,93,93,93,12/12
286b55a2bf,238,238,238,236,236,238,238,11/30
31a2c91c43,242,240,239,240,240,240,239,19/31


## Render-index correspondence on the common set

For a scene: the test views ALL arms share, as render indices (`000NN.png`) per arm,
row-aligned to baseline. Renders at indices NOT in this table have no counterpart in
some arm — skip them when comparing. (Scenes where all arms registered identically
give the trivial full 0..N table.)

In [87]:
def common_view_map(scene):
    names = {a: {n: i for i, n in enumerate(test_names(scene, a))} for a in ARMS}
    shared = [n for n in test_names(scene, BASELINE)
              if all(n in names[a] for a in ARMS)]
    return pd.DataFrame({a: [names[a][n] for n in shared] for a in ARMS})

common_view_map("1ada7a0617")

,baseline,unimodal,gmm,unimodal_all,gmm_all,unimodal_local,gmm_local
0,0,0,0,0,0,0,0
1,1,1,1,1,1,1,1
2,2,2,2,2,2,2,2
3,3,3,3,3,3,3,3
4,4,4,4,4,4,4,4
5,5,5,5,5,5,5,5
6,6,6,6,6,6,6,6
7,7,7,7,7,7,7,7
8,8,8,8,8,8,8,8
9,9,9,9,9,9,9,9


## Aggregate results 

In [88]:
agg = pd.DataFrame(
    {arm: pd.DataFrame([load_results(s, arm) for s in scenes]).mean() for arm in ARMS}
).T[METRICS]
agg.round(3)

,PSNR,SSIM,LPIPS
baseline,30.409,0.922,0.145
unimodal,30.521,0.922,0.143
gmm,30.576,0.923,0.143
unimodal_all,30.662,0.923,0.142
gmm_all,30.627,0.923,0.141
unimodal_local,29.755,0.910,0.158
gmm_local,30.997,0.928,0.138


## Per-view deltas (long table; common test views, keyed by image name)

In [89]:
frames = []
for s in scenes:
    pvs = {a: per_view(s, a).loc[common_imgs[s]] for a in ARMS}
    for arm in ARMS[1:]:
        delta = (pvs[arm] - pvs[BASELINE]).add_prefix("d_")
        delta["scene"], delta["arm"] = s, arm
        frames.append(delta.reset_index())
long = pd.concat(frames, ignore_index=True)
long.shape

(1350, 6)

## Per-scene PSNR deltas — baseline absolute, Δ = arm − baseline (common views)

In [90]:
base_abs = pd.Series(
    {s: per_view(s, BASELINE).loc[common_imgs[s], "PSNR"].mean() for s in scenes},
    name="baseline",
)
tbl = long.groupby(["scene", "arm"])["d_PSNR"].mean().unstack()[ARMS[1:]]
tbl.insert(0, "baseline", base_abs)
tbl = pd.concat([tbl, tbl.agg(["mean", "median"]).set_axis(["MEAN", "MEDIAN"])])
tbl.round(3).style.highlight_max(axis=1, subset=ARMS[1:], props="font-weight: bold;")

arm,baseline,unimodal,gmm,unimodal_all,gmm_all,unimodal_local,gmm_local
0d2ee665be,25.404000,1.950000,2.158000,2.089000,2.031000,2.086000,2.296000
13c3e046d7,34.432000,-0.422000,-0.455000,-0.296000,-0.302000,0.300000,0.265000
1ada7a0617,29.989000,-0.194000,-0.099000,-0.321000,0.046000,-0.041000,0.058000
21d970d8de,26.093000,-0.565000,-0.303000,-0.304000,-0.168000,0.126000,0.143000
25f3b7a318,34.562000,-0.308000,-0.224000,-0.138000,-0.203000,-8.277000,0.227000
27dd4da69e,25.865000,-0.078000,-0.358000,-0.136000,0.012000,-0.128000,0.347000
286b55a2bf,33.364000,0.371000,0.422000,-0.029000,-0.259000,0.016000,-0.066000
31a2c91c43,35.565000,0.261000,0.267000,0.507000,0.348000,0.302000,0.590000
MEAN,30.659000,0.127000,0.176000,0.171000,0.188000,-0.702000,0.483000
MEDIAN,31.677000,-0.136000,-0.162000,-0.137000,-0.078000,0.071000,0.246000


## Win/loss distribution per arm — d_PSNR vs ±0.5 dB

In [91]:
dist = long.groupby("arm")["d_PSNR"].agg(
    win=lambda x: (x > T).sum(),
    neutral=lambda x: x.between(-T, T).sum(),
    loss=lambda x: (x < -T).sum(),
    mean="mean", median="median",
).loc[ARMS[1:]]
dist["win/loss"] = (dist.win / dist.loss).round(2)
dist.round(3)

,win,neutral,loss,mean,median,win/loss
arm,,,,,,
unimodal,32,112,81,-0.037,-0.251,0.40
gmm,33,123,69,0.035,-0.231,0.48
unimodal_all,36,127,62,0.061,-0.185,0.58
gmm_all,39,121,65,0.096,-0.158,0.60
unimodal_local,41,131,53,-1.190,-0.072,0.77
gmm_local,55,148,22,0.434,0.147,2.50


## Unimodal vs gmm — paired per placement (same scene + image)

In [92]:
rows = []
for placement, suffix in [("global", ""), ("all", "_all"), ("local", "_local")]:
    u = long[long.arm == f"unimodal{suffix}"].set_index(["scene", "img"])["d_PSNR"]
    g = long[long.arm == f"gmm{suffix}"].set_index(["scene", "img"])["d_PSNR"]
    diff = (g - u).dropna()
    rows.append({
        "placement": placement, "n": len(diff),
        "gmm_better": (diff > 0).sum(), "uni_better": (diff < 0).sum(),
        "median_diff_dB": diff.median(), "wilcoxon_p": wilcoxon(diff)[1],
    })
pd.DataFrame(rows).set_index("placement").round(4)

,n,gmm_better,uni_better,median_diff_dB,wilcoxon_p
placement,,,,,
global,225,117,108,0.0237,0.0900
all,225,120,105,0.0322,0.4768
local,225,141,84,0.1536,0.0000


## Extreme wins and losses — top/bottom 10 views by d_PSNR

In [96]:
# remove unimodal_local from long table

long = long[long["arm"] != "unimodal_local"]
long = long[long["scene"] != "0d2ee665be"]

cols = ["scene", "img", "arm", "d_PSNR"]
print("WINS")
print(long.nlargest(20, "d_PSNR")[cols].to_string(index=False))
print("\nLOSSES")
print(long.nsmallest(20, "d_PSNR")[cols].to_string(index=False))

WINS
     scene          img          arm   d_PSNR
13c3e046d7 DSC01296.png      gmm_all 7.052940
13c3e046d7 DSC01296.png          gmm 6.985325
13c3e046d7 DSC01296.png unimodal_all 6.866787
13c3e046d7 DSC01296.png     unimodal 6.829002
25f3b7a318 DSC07124.png unimodal_all 6.587280
25f3b7a318 DSC07124.png      gmm_all 6.155697
13c3e046d7 DSC01296.png    gmm_local 5.405453
31a2c91c43 DSC07492.png    gmm_local 4.660347
31a2c91c43 DSC07492.png     unimodal 4.590187
31a2c91c43 DSC07492.png unimodal_all 4.516838
31a2c91c43 DSC07492.png      gmm_all 4.398911
25f3b7a318 DSC07124.png          gmm 4.308321
25f3b7a318 DSC07124.png     unimodal 4.284653
31a2c91c43 DSC07524.png      gmm_all 3.741810
286b55a2bf DSC02334.png          gmm 3.434225
27dd4da69e DSC01149.png    gmm_local 3.423407
31a2c91c43 DSC07524.png unimodal_all 3.322117
25f3b7a318 DSC07124.png    gmm_local 3.095062
31a2c91c43 DSC07492.png          gmm 3.002872
25f3b7a318 DSC07218.png          gmm 2.784088

LOSSES
     scene          i